# Advisor-Focused High-Frequency Notebook

This notebook rewrites the old `5min-0405.ipynb` workflow into a cleaner experiment pipeline that matches the advisor feedback more closely.

## What this notebook covers

1. A **fixed-threshold 5-minute baseline**:
   - keeps the neutral threshold at `0.0002`
   - keeps **class weighting off** by default
   - reports `macro F1`, `macro AUC`, and `per-class F1`

2. A **10-minute extension analysis**, using the **same three-class logic**:
   - rebuilds bars directly from raw intraday text files
   - uses the same neutral threshold `0.0002`
   - tests future **30-minute** and **60-minute** average returns under the same multiclass rule
   - is designed to stay directly comparable with the earlier `5min-0405.ipynb` logic

## Assumptions

- Intraday raw files are expected to look like `EEM.txt`, with `Date,Time,Open,High,Low,Close,Volume` and no header.
- The default run keeps the original `EEM.txt` and adds five liquid ETF files from `50 ETFs 1min`: `DIA`, `EFA`, `EWZ`, `FXI`, and `GLD`.
- Before any model training starts, the notebook audits every selected file for full time span, row count, timestamp parsing, duplicates, OHLC consistency, non-positive prices, and negative volume.

In [ ]:
import copy
import json
import math
import os
import random
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler, label_binarize

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("talk")

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "hf_advisor_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ---------------------------
# User-facing configuration
# ---------------------------
RAW_FILE_GLOBS = ["*.txt"]
RAW_DATA_DIRS = [PROJECT_ROOT]
ETF_1MIN_DIR = PROJECT_ROOT / "50 ETFs 1min"
SELECTED_ADDITIONAL_ETFS = []  # Add symbols only after supplying their raw files.
EXPLICIT_ASSET_FILES = ["EEM.txt"] + [
    str(Path("50 ETFs 1min") / f"{symbol}.txt")
    for symbol in SELECTED_ADDITIONAL_ETFS
]
MAX_ASSETS = None          # set an int to limit runtime while testing
MAX_ROWS_PER_ASSET = 0     # 0 means use all rows
REQUIRE_CLEAN_AUDIT = True

SESSION_START = "09:30"
SESSION_END = "16:00"

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
LOOKBACK_MINUTES = 300  # keep the same 5-hour history window as before
BATCH_SIZE = 32
EPOCHS = 20
PATIENCE = 5
WEIGHT_DECAY = 1e-5
USE_CLASS_WEIGHT = False

MODELS_TO_RUN = ["TCN", "DLinear", "LSTM"]
MODEL_LR = {
    "TCN": 1e-4,
    "DLinear": 1e-5,
    "LSTM": 1e-4,
}

RUN_FIXED_THRESHOLD_BASELINE = True
RUN_10MIN_MULTICLASS_EXTENSION = True

FIXED_THRESHOLD = 0.0002

EXPERIMENTS = []

if RUN_FIXED_THRESHOLD_BASELINE:
    EXPERIMENTS.append(
        {
            "experiment_name": "baseline_5min_multiclass_thr_0p0002_60m",
            "task": "multiclass_threshold",
            "bar_minutes": 5,
            "horizon_minutes": 60,
            "neutral_threshold": FIXED_THRESHOLD,
            "class_names": ["down", "neutral", "up"],
        }
    )

if RUN_10MIN_MULTICLASS_EXTENSION:
    EXPERIMENTS.extend(
        [
            {
                "experiment_name": "extension_10min_multiclass_thr_0p0002_30m",
                "task": "multiclass_threshold",
                "bar_minutes": 10,
                "horizon_minutes": 30,
                "neutral_threshold": FIXED_THRESHOLD,
                "class_names": ["down", "neutral", "up"],
            },
            {
                "experiment_name": "extension_10min_multiclass_thr_0p0002_60m",
                "task": "multiclass_threshold",
                "bar_minutes": 10,
                "horizon_minutes": 60,
                "neutral_threshold": FIXED_THRESHOLD,
                "class_names": ["down", "neutral", "up"],
            },
        ]
    )

print("Configured experiments:")
for exp in EXPERIMENTS:
    print(
        f"  - {exp['experiment_name']} | task={exp['task']} | "
        f"bar={exp['bar_minutes']}m | horizon={exp['horizon_minutes']}m"
    )

In [ ]:
def resolve_asset_path(path_like: str | Path) -> Path:
    path = Path(path_like)
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    return path


def discover_intraday_files() -> list[Path]:
    if EXPLICIT_ASSET_FILES:
        files = [resolve_asset_path(name) for name in EXPLICIT_ASSET_FILES]
    else:
        files = []
        for data_dir in RAW_DATA_DIRS:
            for pattern in RAW_FILE_GLOBS:
                files.extend(data_dir.glob(pattern))

    deduped = []
    seen = set()
    ordered_files = files if EXPLICIT_ASSET_FILES else sorted(files, key=lambda p: str(p))
    for path in ordered_files:
        if path.is_dir():
            continue
        if path.suffix.lower() != ".txt":
            continue
        if not path.exists():
            raise FileNotFoundError(f"Configured intraday file does not exist: {path}")
        resolved = path.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        deduped.append(path)

    if MAX_ASSETS is not None:
        deduped = deduped[:MAX_ASSETS]
    return deduped


def infer_symbol(path: Path) -> str:
    return path.stem.strip()


def load_intraday_txt(path: Path, max_rows: int = 0) -> pd.DataFrame:
    df = pd.read_csv(
        path,
        header=None,
        names=["Date", "Time", "Open", "High", "Low", "Close", "Volume"],
    )
    dt = pd.to_datetime(df["Date"] + " " + df["Time"], format="%m/%d/%Y %H:%M")
    df.index = dt
    df = df.drop(columns=["Date", "Time"]).sort_index()
    if max_rows > 0:
        df = df.iloc[:max_rows].copy()
    return df


def audit_intraday_file(path: Path) -> dict:
    symbol = infer_symbol(path)
    if not path.exists():
        return {
            "symbol": symbol,
            "path": str(path),
            "status": "missing",
            "rows": 0,
            "issues": "file does not exist",
        }

    raw = pd.read_csv(
        path,
        header=None,
        names=["Date", "Time", "Open", "High", "Low", "Close", "Volume"],
    )
    dt = pd.to_datetime(
        raw["Date"] + " " + raw["Time"],
        format="%m/%d/%Y %H:%M",
        errors="coerce",
    )
    numeric = raw[["Open", "High", "Low", "Close", "Volume"]].apply(pd.to_numeric, errors="coerce")

    start_minutes = int(SESSION_START[:2]) * 60 + int(SESSION_START[3:])
    end_minutes = int(SESSION_END[:2]) * 60 + int(SESSION_END[3:])
    minute_of_day = dt.dt.hour * 60 + dt.dt.minute
    outside_session = dt.notna() & ((minute_of_day < start_minutes) | (minute_of_day > end_minutes))

    bad_ohlc = (
        (numeric["High"] < numeric[["Open", "Close", "Low"]].max(axis=1))
        | (numeric["Low"] > numeric[["Open", "Close", "High"]].min(axis=1))
    )

    issue_counts = {
        "parse_nulls": int(dt.isna().sum()),
        "duplicate_timestamps": int(dt.duplicated().sum()),
        "numeric_nulls": int(numeric.isna().sum().sum()),
        "nonpositive_ohlc_rows": int((numeric[["Open", "High", "Low", "Close"]] <= 0).any(axis=1).sum()),
        "negative_volume_rows": int((numeric["Volume"] < 0).sum()),
        "bad_ohlc_rows": int(bad_ohlc.sum()),
        "raw_nonmonotonic_steps": int((dt.diff().dropna() < pd.Timedelta(0)).sum()),
    }
    severe_issues = {key: value for key, value in issue_counts.items() if value > 0}

    return {
        "symbol": symbol,
        "path": str(path),
        "status": "ok" if not severe_issues else "check",
        "rows": int(len(raw)),
        "start": dt.min(),
        "end": dt.max(),
        "unique_days": int(dt.dt.date.nunique()),
        "outside_regular_session_rows": int(outside_session.sum()),
        **issue_counts,
        "issues": "; ".join(f"{key}={value}" for key, value in severe_issues.items()),
    }


def audit_asset_files(asset_files: list[Path]) -> pd.DataFrame:
    audit_rows = [audit_intraday_file(path) for path in asset_files]
    audit_df = pd.DataFrame(audit_rows)
    if len(audit_df) == 0:
        return audit_df
    audit_df["start"] = pd.to_datetime(audit_df["start"])
    audit_df["end"] = pd.to_datetime(audit_df["end"])
    return audit_df


def assert_asset_audit_clean(audit_df: pd.DataFrame) -> None:
    if not REQUIRE_CLEAN_AUDIT or len(audit_df) == 0:
        return
    bad_rows = audit_df[audit_df["status"] != "ok"]
    if len(bad_rows) > 0:
        display_cols = ["symbol", "status", "issues", "path"]
        raise ValueError(
            "Data audit found issues before training:\n"
            + bad_rows[display_cols].to_string(index=False)
        )


def json_default(obj):
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    raise TypeError(f"Object of type {obj.__class__.__name__} is not JSON serializable")


def _resample_one_day(day_df: pd.DataFrame, bar_minutes: int) -> pd.DataFrame:
    freq = f"{bar_minutes}min"
    resampled = (
        day_df.resample(freq, label="right", closed="right")
        .agg(
            {
                "Open": "first",
                "High": "max",
                "Low": "min",
                "Close": "last",
                "Volume": "sum",
            }
        )
        .sort_index()
    )

    session_date = pd.Timestamp(day_df.index[0].date())
    session_start = session_date + pd.Timedelta(SESSION_START + ":00")
    session_end = session_date + pd.Timedelta(SESSION_END + ":00")
    valid_index = pd.date_range(
        start=session_start + pd.Timedelta(minutes=bar_minutes),
        end=session_end,
        freq=freq,
    )
    resampled = resampled.reindex(valid_index)

    first_valid = resampled["Close"].first_valid_index()
    if first_valid is None:
        return pd.DataFrame(columns=resampled.columns)

    resampled = resampled.loc[first_valid:].copy()

    price_cols = ["Open", "High", "Low", "Close"]
    resampled[price_cols] = resampled[price_cols].ffill()
    resampled["Volume"] = resampled["Volume"].fillna(0.0)

    # If the first valid row still has partial missing OHLC fields, use Close to patch it.
    first_close = resampled["Close"].iloc[0]
    for col in price_cols:
        if pd.isna(resampled.iloc[0][col]):
            resampled.iloc[0, resampled.columns.get_loc(col)] = first_close

    return resampled.dropna()


def resample_intraday_bars(ticks: pd.DataFrame, bar_minutes: int) -> pd.DataFrame:
    by_day = []
    for _, day_df in ticks.groupby(ticks.index.date):
        session_df = day_df.between_time(SESSION_START, SESSION_END)
        if len(session_df) == 0:
            continue
        bars = _resample_one_day(session_df, bar_minutes)
        if len(bars) > 0:
            by_day.append(bars)
    if not by_day:
        return pd.DataFrame(columns=["Open", "High", "Low", "Close", "Volume"])
    return pd.concat(by_day).sort_index()


asset_files = discover_intraday_files()
detected_symbols = [infer_symbol(path) for path in asset_files]
print("Detected intraday files:", detected_symbols)

asset_audit_df = audit_asset_files(asset_files)
audit_path = OUTPUT_DIR / "advisor_hf_asset_audit.csv"
asset_audit_df.to_csv(audit_path, index=False)
print(f"Saved asset audit to: {audit_path}")
display(asset_audit_df)
assert_asset_audit_clean(asset_audit_df)

In [ ]:
def compute_rsi(close: pd.Series, window: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)
    avg_gain = gain.ewm(alpha=1 / window, adjust=False, min_periods=window).mean()
    avg_loss = loss.ewm(alpha=1 / window, adjust=False, min_periods=window).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    return rsi.fillna(method="bfill")


def compute_stochastic(high: pd.Series, low: pd.Series, close: pd.Series, window: int = 14, smooth_d: int = 3) -> tuple[pd.Series, pd.Series]:
    rolling_low = low.rolling(window=window, min_periods=window).min()
    rolling_high = high.rolling(window=window, min_periods=window).max()
    denom = (rolling_high - rolling_low).replace(0, np.nan)
    stoch_k = 100 * (close - rolling_low) / denom
    stoch_d = stoch_k.rolling(window=smooth_d, min_periods=smooth_d).mean()
    return stoch_k.fillna(method="bfill"), stoch_d.fillna(method="bfill")


def compute_atr(high: pd.Series, low: pd.Series, close: pd.Series, window: int = 14) -> pd.Series:
    prev_close = close.shift(1)
    tr = pd.concat(
        [
            (high - low).abs(),
            (high - prev_close).abs(),
            (low - prev_close).abs(),
        ],
        axis=1,
    ).max(axis=1)
    atr = tr.rolling(window=window, min_periods=window).mean()
    return atr.fillna(method="bfill")


def compute_macd(close: pd.Series, fast: int = 12, slow: int = 26, signal: int = 9) -> tuple[pd.Series, pd.Series, pd.Series]:
    ema_fast = close.ewm(span=fast, adjust=False).mean()
    ema_slow = close.ewm(span=slow, adjust=False).mean()
    macd = ema_fast - ema_slow
    macd_signal = macd.ewm(span=signal, adjust=False).mean()
    macd_hist = macd - macd_signal
    return macd, macd_signal, macd_hist


def compute_volume_features(volume: pd.Series, close: pd.Series, window: int = 20) -> tuple[pd.Series, pd.Series, pd.Series]:
    volume_ma = volume.rolling(window=window, min_periods=window).mean()
    volume_ratio = volume / volume_ma.replace(0, np.nan)
    direction = np.sign(close.diff().fillna(0.0))
    obv = (direction * volume).fillna(0.0).cumsum()
    return (
        volume_ma.fillna(method="bfill"),
        volume_ratio.replace([np.inf, -np.inf], np.nan).fillna(method="bfill"),
        obv,
    )


def build_feature_df_from_bars(bars: pd.DataFrame) -> pd.DataFrame:
    if len(bars) == 0:
        return pd.DataFrame()

    df = bars.copy()
    df["LogReturn"] = df.groupby(df.index.date)["Close"].transform(lambda s: np.log(s).diff())
    df["RSI"] = compute_rsi(df["Close"], window=14)
    df["Stoch_K"], df["Stoch_D"] = compute_stochastic(df["High"], df["Low"], df["Close"], window=14, smooth_d=3)
    df["ATR"] = compute_atr(df["High"], df["Low"], df["Close"], window=14)
    df["MACD"], df["MACD_Signal"], df["MACD_Histogram"] = compute_macd(df["Close"])
    df["Volume_MA"], df["Volume_Ratio"], df["OBV"] = compute_volume_features(df["Volume"], df["Close"], window=20)

    feature_df = df[
        [
            "LogReturn",
            "RSI",
            "Stoch_K",
            "Stoch_D",
            "ATR",
            "MACD",
            "MACD_Signal",
            "MACD_Histogram",
            "Volume_MA",
            "Volume_Ratio",
            "OBV",
        ]
    ].replace([np.inf, -np.inf], np.nan)

    feature_df = feature_df.dropna().copy()
    return feature_df


def prepare_feature_dataset(path: Path, bar_minutes: int) -> pd.DataFrame:
    ticks = load_intraday_txt(path, max_rows=MAX_ROWS_PER_ASSET)
    bars = resample_intraday_bars(ticks, bar_minutes=bar_minutes)
    features = build_feature_df_from_bars(bars)
    return features

In [ ]:
def split_by_time(df: pd.DataFrame, train_ratio: float = 0.70, val_ratio: float = 0.15) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    total = len(df)
    train_end = int(total * train_ratio)
    val_end = train_end + int(total * val_ratio)
    train_df = df.iloc[:train_end].copy()
    val_df = df.iloc[train_end:val_end].copy()
    test_df = df.iloc[val_end:].copy()
    return train_df, val_df, test_df


def is_same_trading_day(index_slice: pd.Index) -> bool:
    if len(index_slice) == 0:
        return False
    dates = pd.Series(index_slice).dt.date
    return dates.nunique() == 1


def create_sequences(
    data: pd.DataFrame,
    lookback_bars: int,
    horizon_bars: int,
    task: str,
    neutral_threshold: float | None = None,
) -> tuple[np.ndarray, np.ndarray, list[pd.Timestamp], dict]:
    feature_columns = [col for col in data.columns if col != "LogReturn"]
    features = data[feature_columns].values
    returns = data["LogReturn"].values
    indices = data.index

    X, y, sample_times = [], [], []
    cross_day_skipped = 0

    for i in range(lookback_bars, len(data) - horizon_bars):
        input_times = indices[i - lookback_bars : i]
        future_times = indices[i : i + horizon_bars]
        combined_times = indices[i - lookback_bars : i + horizon_bars]

        if not is_same_trading_day(input_times) or not is_same_trading_day(future_times) or not is_same_trading_day(combined_times):
            cross_day_skipped += 1
            continue

        future_avg_return = float(np.mean(returns[i : i + horizon_bars]))
        if math.isnan(future_avg_return):
            continue

        if task == "multiclass_threshold":
            if neutral_threshold is None:
                raise ValueError("neutral_threshold is required for multiclass_threshold")
            if future_avg_return < -neutral_threshold:
                label = 0
            elif future_avg_return > neutral_threshold:
                label = 2
            else:
                label = 1
        else:
            raise ValueError(f"Unsupported task: {task}")

        X.append(features[i - lookback_bars : i])
        y.append(label)
        sample_times.append(indices[i])

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.int64)
    stats = {
        "total": int(len(y)),
        "cross_day_skipped": int(cross_day_skipped),
    }

    if len(y) > 0:
        class_counts = pd.Series(y).value_counts().sort_index()
        for cls, count in class_counts.items():
            stats[f"class_{cls}_count"] = int(count)
            stats[f"class_{cls}_ratio"] = float(count / len(y))

    return X, y, sample_times, stats


def standardize_sequence_splits(X_train_raw: np.ndarray, X_val_raw: np.ndarray, X_test_raw: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    n_features = X_train_raw.shape[-1]
    train_steps = X_train_raw.shape[1]

    train_flat = X_train_raw.reshape(-1, n_features)
    val_flat = X_val_raw.reshape(-1, n_features)
    test_flat = X_test_raw.reshape(-1, n_features)

    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_flat).reshape(X_train_raw.shape[0], train_steps, n_features)
    val_scaled = scaler.transform(val_flat).reshape(X_val_raw.shape[0], train_steps, n_features)
    test_scaled = scaler.transform(test_flat).reshape(X_test_raw.shape[0], train_steps, n_features)
    return train_scaled, val_scaled, test_scaled

In [ ]:
def softmax_np(logits: np.ndarray) -> np.ndarray:
    logits = logits - np.max(logits, axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / exp_logits.sum(axis=1, keepdims=True)


def calculate_classification_metrics(logits: np.ndarray, targets: np.ndarray, class_names: list[str]) -> dict:
    probs = softmax_np(logits)
    preds = np.argmax(probs, axis=1)
    num_classes = len(class_names)
    labels = list(range(num_classes))

    metrics = {
        "accuracy": float(accuracy_score(targets, preds)),
        "precision_macro": float(precision_score(targets, preds, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(targets, preds, average="macro", zero_division=0)),
        "f1_macro": float(f1_score(targets, preds, average="macro", zero_division=0)),
        "f1_weighted": float(f1_score(targets, preds, average="weighted", zero_division=0)),
        "confusion_matrix": confusion_matrix(targets, preds, labels=labels).tolist(),
    }

    precision_per_class = precision_score(targets, preds, average=None, labels=labels, zero_division=0)
    recall_per_class = recall_score(targets, preds, average=None, labels=labels, zero_division=0)
    f1_per_class = f1_score(targets, preds, average=None, labels=labels, zero_division=0)

    for i, class_name in enumerate(class_names):
        metrics[f"{class_name}_precision"] = float(precision_per_class[i])
        metrics[f"{class_name}_recall"] = float(recall_per_class[i])
        metrics[f"{class_name}_f1"] = float(f1_per_class[i])

    try:
        target_binarized = label_binarize(targets, classes=labels)
        auc_per_class = []
        for class_id in labels:
            auc_per_class.append(
                roc_auc_score(target_binarized[:, class_id], probs[:, class_id])
            )
        metrics["auc_macro"] = float(np.mean(auc_per_class))
        for i, class_name in enumerate(class_names):
            metrics[f"{class_name}_auc"] = float(auc_per_class[i])
    except ValueError:
        metrics["auc_macro"] = float("nan")
        for class_name in class_names:
            metrics[f"{class_name}_auc"] = float("nan")

    return metrics


class TCNClassifier(nn.Module):
    def __init__(self, input_size: int, num_classes: int, num_channels: list[int] | None = None, kernel_size: int = 3, dropout: float = 0.2):
        super().__init__()
        num_channels = num_channels or [32, 32, 32]
        self.conv1 = nn.Conv1d(input_size, num_channels[0], kernel_size, padding=(kernel_size - 1) // 2)
        self.bn1 = nn.BatchNorm1d(num_channels[0])
        self.conv2 = nn.Conv1d(num_channels[0], num_channels[1], kernel_size, padding=(kernel_size - 1) // 2)
        self.bn2 = nn.BatchNorm1d(num_channels[1])
        self.conv3 = nn.Conv1d(num_channels[1], num_channels[2], kernel_size, padding=(kernel_size - 1) // 2)
        self.bn3 = nn.BatchNorm1d(num_channels[2])
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(num_channels[2], num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.transpose(1, 2)
        x = self.dropout(self.relu(self.bn1(self.conv1(x))))
        x = self.dropout(self.relu(self.bn2(self.conv2(x))))
        x = self.dropout(self.relu(self.bn3(self.conv3(x))))
        x = self.pool(x).squeeze(-1)
        return self.fc(x)


class DLinearClassifier(nn.Module):
    def __init__(self, seq_len: int, num_features: int, num_classes: int, dropout: float = 0.2):
        super().__init__()
        self.linear1 = nn.Linear(seq_len * num_features, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.linear2 = nn.Linear(256, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.linear3 = nn.Linear(128, 64)
        self.bn3 = nn.BatchNorm1d(64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.output = nn.Linear(64, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.reshape(x.size(0), -1)
        x = self.dropout(self.relu(self.bn1(self.linear1(x))))
        x = self.dropout(self.relu(self.bn2(self.linear2(x))))
        x = self.dropout(self.relu(self.bn3(self.linear3(x))))
        return self.output(x)


class LSTMClassifier(nn.Module):
    def __init__(self, input_size: int, num_classes: int, hidden_size: int = 64, num_layers: int = 2, dropout: float = 0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.lstm(x)
        out = self.dropout(out[:, -1, :])
        return self.fc(out)


def build_model(model_name: str, seq_len: int, num_features: int, num_classes: int) -> nn.Module:
    if model_name == "TCN":
        return TCNClassifier(input_size=num_features, num_classes=num_classes)
    if model_name == "DLinear":
        return DLinearClassifier(seq_len=seq_len, num_features=num_features, num_classes=num_classes)
    if model_name == "LSTM":
        return LSTMClassifier(input_size=num_features, num_classes=num_classes)
    raise ValueError(f"Unsupported model: {model_name}")


def init_weights_safe(module: nn.Module) -> None:
    if isinstance(module, nn.Linear):
        nn.init.xavier_uniform_(module.weight, gain=0.5)
        if module.bias is not None:
            nn.init.constant_(module.bias, 0.01)
    elif isinstance(module, nn.Conv1d):
        nn.init.xavier_uniform_(module.weight, gain=0.5)
        if module.bias is not None:
            nn.init.constant_(module.bias, 0.01)
    elif isinstance(module, nn.LSTM):
        for name, param in module.named_parameters():
            if "weight" in name:
                nn.init.xavier_uniform_(param, gain=0.5)
            elif "bias" in name:
                nn.init.constant_(param, 0.01)

In [ ]:
def make_loader(X: np.ndarray, y: np.ndarray, batch_size: int, shuffle: bool = False) -> DataLoader:
    dataset = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.long),
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=0)


def build_loss_fn(num_classes: int, y_train: np.ndarray, use_class_weight: bool) -> nn.Module:
    if not use_class_weight:
        return nn.CrossEntropyLoss()
    class_counts = np.bincount(y_train, minlength=num_classes)
    safe_counts = np.where(class_counts == 0, 1, class_counts)
    weights = len(y_train) / (num_classes * safe_counts)
    return nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32, device=DEVICE))


def evaluate_model(model: nn.Module, loader: DataLoader) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    logits_list = []
    target_list = []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            logits = model(X_batch)
            logits_list.append(logits.detach().cpu().numpy())
            target_list.append(y_batch.detach().cpu().numpy())
    return np.concatenate(logits_list, axis=0), np.concatenate(target_list, axis=0)


def train_model(
    model: nn.Module,
    model_name: str,
    train_loader: DataLoader,
    val_loader: DataLoader,
    y_train: np.ndarray,
    class_names: list[str],
    lr: float,
    epochs: int,
    patience: int,
    use_class_weight: bool,
) -> tuple[nn.Module, dict, list[dict]]:
    model = model.to(DEVICE)
    loss_fn = build_loss_fn(num_classes=len(class_names), y_train=y_train, use_class_weight=use_class_weight)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

    best_state = None
    best_metrics = None
    best_score = -float("inf")
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            optimizer.zero_grad()
            logits = model(X_batch)
            loss = loss_fn(logits, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()

        val_logits, val_targets = evaluate_model(model, val_loader)
        val_metrics = calculate_classification_metrics(val_logits, val_targets, class_names)
        mean_train_loss = train_loss / max(len(train_loader), 1)
        scheduler.step(mean_train_loss)

        record = {
            "epoch": epoch,
            "train_loss": float(mean_train_loss),
            "val_f1_macro": float(val_metrics["f1_macro"]),
            "val_auc_macro": float(val_metrics["auc_macro"]) if not math.isnan(val_metrics["auc_macro"]) else np.nan,
        }
        history.append(record)

        score = val_metrics["f1_macro"]
        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())
            best_metrics = val_metrics
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epoch == 1 or epoch % 5 == 0:
            print(
                f"    [{model_name}] epoch={epoch:02d} "
                f"train_loss={mean_train_loss:.4f} "
                f"val_macro_f1={val_metrics['f1_macro']:.4f} "
                f"val_macro_auc={val_metrics['auc_macro']:.4f}"
            )

        if epochs_without_improvement >= patience:
            print(f"    [{model_name}] early stop at epoch {epoch}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_metrics, history

In [ ]:
feature_cache: dict[tuple[str, int], pd.DataFrame] = {}


def get_feature_df(path: Path, bar_minutes: int) -> pd.DataFrame:
    key = (str(path), bar_minutes)
    if key not in feature_cache:
        feature_cache[key] = prepare_feature_dataset(path, bar_minutes)
    return feature_cache[key].copy()


def run_single_experiment(path: Path, experiment: dict) -> tuple[list[dict], list[dict]]:
    symbol = infer_symbol(path)
    feature_df = get_feature_df(path, bar_minutes=experiment["bar_minutes"])

    if len(feature_df) == 0:
        print(f"Skip {symbol} | {experiment['experiment_name']}: no feature rows.")
        return [], []

    lookback_bars = LOOKBACK_MINUTES // experiment["bar_minutes"]
    horizon_bars = experiment["horizon_minutes"] // experiment["bar_minutes"]

    train_df, val_df, test_df = split_by_time(feature_df, train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO)

    X_train_raw, y_train, train_times, train_stats = create_sequences(
        train_df,
        lookback_bars=lookback_bars,
        horizon_bars=horizon_bars,
        task=experiment["task"],
        neutral_threshold=experiment["neutral_threshold"],
    )
    X_val_raw, y_val, val_times, val_stats = create_sequences(
        val_df,
        lookback_bars=lookback_bars,
        horizon_bars=horizon_bars,
        task=experiment["task"],
        neutral_threshold=experiment["neutral_threshold"],
    )
    X_test_raw, y_test, test_times, test_stats = create_sequences(
        test_df,
        lookback_bars=lookback_bars,
        horizon_bars=horizon_bars,
        task=experiment["task"],
        neutral_threshold=experiment["neutral_threshold"],
    )

    if min(len(y_train), len(y_val), len(y_test)) == 0:
        print(f"Skip {symbol} | {experiment['experiment_name']}: one split has zero samples.")
        return [], []

    X_train, X_val, X_test = standardize_sequence_splits(X_train_raw, X_val_raw, X_test_raw)

    train_loader = make_loader(X_train, y_train, batch_size=BATCH_SIZE, shuffle=False)
    val_loader = make_loader(X_val, y_val, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = make_loader(X_test, y_test, batch_size=BATCH_SIZE, shuffle=False)

    label_records = []
    for split_name, stats in [("train", train_stats), ("val", val_stats), ("test", test_stats)]:
        row = {
            "symbol": symbol,
            "experiment_name": experiment["experiment_name"],
            "task": experiment["task"],
            "bar_minutes": experiment["bar_minutes"],
            "horizon_minutes": experiment["horizon_minutes"],
            "split": split_name,
            **stats,
        }
        label_records.append(row)

    run_results = []
    for model_name in MODELS_TO_RUN:
        print(
            f"Running {symbol} | {experiment['experiment_name']} | "
            f"{model_name} | lookback={lookback_bars} bars | horizon={horizon_bars} bars"
        )
        model = build_model(
            model_name=model_name,
            seq_len=lookback_bars,
            num_features=X_train.shape[-1],
            num_classes=len(experiment["class_names"]),
        )
        model.apply(init_weights_safe)

        trained_model, best_val_metrics, history = train_model(
            model=model,
            model_name=model_name,
            train_loader=train_loader,
            val_loader=val_loader,
            y_train=y_train,
            class_names=experiment["class_names"],
            lr=MODEL_LR[model_name],
            epochs=EPOCHS,
            patience=PATIENCE,
            use_class_weight=USE_CLASS_WEIGHT,
        )

        val_logits, val_targets = evaluate_model(trained_model, val_loader)
        test_logits, test_targets = evaluate_model(trained_model, test_loader)

        val_metrics = calculate_classification_metrics(val_logits, val_targets, experiment["class_names"])
        test_metrics = calculate_classification_metrics(test_logits, test_targets, experiment["class_names"])

        run_row = {
            "symbol": symbol,
            "model": model_name,
            "experiment_name": experiment["experiment_name"],
            "task": experiment["task"],
            "bar_minutes": experiment["bar_minutes"],
            "horizon_minutes": experiment["horizon_minutes"],
            "lookback_minutes": LOOKBACK_MINUTES,
            "lookback_bars": lookback_bars,
            "horizon_bars": horizon_bars,
            "neutral_threshold": experiment["neutral_threshold"],
            "use_class_weight": USE_CLASS_WEIGHT,
            "train_samples": int(len(y_train)),
            "val_samples": int(len(y_val)),
            "test_samples": int(len(y_test)),
        }

        for prefix, metric_dict in [("val", val_metrics), ("test", test_metrics)]:
            for key, value in metric_dict.items():
                if key == "confusion_matrix":
                    run_row[f"{prefix}_{key}"] = json.dumps(value)
                else:
                    run_row[f"{prefix}_{key}"] = value

        run_results.append(run_row)

    return run_results, label_records


def run_all_experiments(asset_files: list[Path], experiments: list[dict]) -> tuple[pd.DataFrame, pd.DataFrame]:
    all_results = []
    all_label_records = []

    for path in asset_files:
        symbol = infer_symbol(path)
        print("\n" + "=" * 100)
        print(f"Asset: {symbol}")
        if "asset_audit_df" in globals() and len(asset_audit_df) > 0:
            audit_match = asset_audit_df[asset_audit_df["symbol"] == symbol]
            if len(audit_match) > 0:
                audit_row = audit_match.iloc[0]
                print(
                    "Data audit: "
                    f"status={audit_row['status']} | "
                    f"rows={audit_row['rows']:,} | "
                    f"span={audit_row['start']} -> {audit_row['end']} | "
                    f"unique_days={audit_row['unique_days']:,} | "
                    f"outside_regular_session_rows={audit_row['outside_regular_session_rows']:,}"
                )
        print("=" * 100)
        for experiment in experiments:
            result_rows, label_rows = run_single_experiment(path, experiment)
            all_results.extend(result_rows)
            all_label_records.extend(label_rows)

    results_df = pd.DataFrame(all_results)
    label_df = pd.DataFrame(all_label_records)
    return results_df, label_df

In [ ]:
if not asset_files:
    raise FileNotFoundError(
        "No configured intraday txt files were detected. "
        "Check EXPLICIT_ASSET_FILES, RAW_DATA_DIRS, and the 50 ETFs 1min folder."
    )

results_df, label_balance_df = run_all_experiments(asset_files, EXPERIMENTS)

if len(results_df) == 0:
    raise RuntimeError("No experiment results were produced. Check whether sequence creation returned empty splits.")

results_path = OUTPUT_DIR / "advisor_hf_results.csv"
label_path = OUTPUT_DIR / "advisor_hf_label_balance.csv"
results_df.to_csv(results_path, index=False)
label_balance_df.to_csv(label_path, index=False)

print(f"Saved results to: {results_path}")
print(f"Saved label balance to: {label_path}")

In [ ]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

summary_cols = [
    "symbol",
    "experiment_name",
    "model",
    "val_f1_macro",
    "val_auc_macro",
    "test_f1_macro",
    "test_auc_macro",
    "test_accuracy",
]
print("Full experiment summary:")
display(results_df[summary_cols].sort_values(["experiment_name", "symbol", "val_f1_macro"], ascending=[True, True, False]))

best_runs_df = (
    results_df.sort_values(["experiment_name", "symbol", "val_f1_macro"], ascending=[True, True, False])
    .groupby(["experiment_name", "symbol"], as_index=False)
    .first()
)

best_runs_path = OUTPUT_DIR / "advisor_hf_best_runs.csv"
best_runs_df.to_csv(best_runs_path, index=False)
print(f"Saved best-per-asset summary to: {best_runs_path}")

metric_focus_cols = [col for col in best_runs_df.columns if col.startswith("test_") and ("_f1" in col or "_auc" in col)]
print("Best runs with advisor-requested metrics:")
display(best_runs_df[["symbol", "experiment_name", "model"] + metric_focus_cols].sort_values(["experiment_name", "symbol"]))

if len(best_runs_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    plot_df = best_runs_df.copy()
    sns.barplot(
        data=plot_df,
        x="experiment_name",
        y="test_f1_macro",
        hue="model",
        ax=axes[0],
    )
    axes[0].set_title("Best Test Macro F1")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("Macro F1")
    axes[0].tick_params(axis="x", rotation=20)

    sns.barplot(
        data=plot_df,
        x="experiment_name",
        y="test_auc_macro",
        hue="model",
        ax=axes[1],
    )
    axes[1].set_title("Best Test Macro AUC")
    axes[1].set_xlabel("")
    axes[1].set_ylabel("Macro AUC")
    axes[1].tick_params(axis="x", rotation=20)

    plt.tight_layout()
    plt.show()

payload = {
    "config": {
        "lookback_minutes": LOOKBACK_MINUTES,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "patience": PATIENCE,
        "use_class_weight": USE_CLASS_WEIGHT,
        "models_to_run": MODELS_TO_RUN,
        "detected_assets": [infer_symbol(path) for path in asset_files],
        "selected_additional_etfs": SELECTED_ADDITIONAL_ETFS,
    },
    "experiments": EXPERIMENTS,
    "asset_audit": asset_audit_df.to_dict(orient="records"),
    "best_runs": best_runs_df.to_dict(orient="records"),
}
with open(OUTPUT_DIR / "advisor_hf_best_runs.json", "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, ensure_ascii=False, default=json_default)

print("\nImportant note:")
print(
    "  The archive defaults to EEM; additional ETF files can be supplied under "
    f"`50 ETFs 1min`: {SELECTED_ADDITIONAL_ETFS}. "
    "Each file is audited before model training; see advisor_hf_asset_audit.csv."
)

## Draft Introduction

Financial return prediction at high frequency is difficult because the signal-to-noise ratio is low and short-horizon labels can be unstable. In our earlier 5-minute experiments, a threshold sweep showed that performance was highly sensitive to the definition of the neutral class, suggesting that label construction may matter at least as much as model architecture. In particular, the sweep indicated that a neutral threshold of `0.0002` produces a more balanced target than larger thresholds, which quickly make the neutral class dominant.

Motivated by this finding, this paper studies two related questions. First, after fixing the neutral threshold at `0.0002`, we ask whether the original 5-minute multiclass setup remains stable across additional liquid assets when evaluated with macro-sensitive metrics rather than headline accuracy alone. Second, we ask whether 10-minute aggregation can improve the same multiclass prediction problem when the future target is defined over smoother 30-minute and 60-minute average-return windows. This design keeps the label rule unchanged while changing the bar frequency and prediction horizon, making the comparison more directly interpretable.

The broader goal is not to claim that one neural architecture dominates in all market conditions, but to understand how bar aggregation and target design shape apparent predictive performance in high-frequency financial classification tasks.

## Draft Related Work

A large literature studies financial time-series prediction with both traditional statistical models and deep learning methods. Within the deep-learning branch, recurrent models such as LSTM are often used to capture sequential dependence, while convolution-based sequence models such as TCN aim to model temporal patterns with shorter optimization paths. More recently, lightweight linear architectures have also been considered competitive baselines in time-series forecasting because they are simple, stable, and sometimes surprisingly strong.

A second relevant strand of literature focuses on market microstructure noise and label construction. In high-frequency settings, predictive results can change substantially depending on whether the target is defined as a point return, a cumulative return, an average future return, or a thresholded directional label. This issue is especially important in multiclass setups with a neutral category, because a poorly chosen threshold can create severe class imbalance and distort apparent performance.

Our work is closest to studies that compare model families under alternative target definitions rather than treating the label as fixed. The contribution here is practical and empirical: we show how evaluation changes once the neutral threshold is fixed at a more balanced level, and we test whether 10-minute aggregation with smoother 30-minute and 60-minute average-return horizons yields more stable multiclass classification behavior under the same threshold rule.

Citation placeholders to fill later:

- LSTM for sequence modeling in finance
- TCN for temporal convolution on sequential data
- Linear baselines for time-series forecasting
- High-frequency noise / label engineering / market microstructure

## Draft Method

### Data and preprocessing

We start from raw intraday trade bars stored in text format and reconstruct regular intraday bars inside the U.S. cash-equity session from `09:30` to `16:00`. For each asset, we create both 5-minute and 10-minute bars directly from the raw input so that the extension analysis is based on genuine aggregation rather than post-hoc averaging of precomputed indicators. Missing within-day bars are forward-filled in price fields after the first observed bar of the session, while missing volume is set to zero.

### Features

For each regularized intraday series, we compute a feature set consisting of log return, RSI, stochastic oscillator (`%K`, `%D`), ATR, MACD, MACD signal, MACD histogram, volume moving average, volume ratio, and OBV. These features match the general design of the earlier notebook, which helps keep the new experiments comparable to the previous threshold-sweep results.

### Targets

We evaluate three directly comparable multiclass settings.

1. **Fixed-threshold 5-minute multiclass baseline.**
   The target is the average future log return over the next 60 minutes, using 5-minute bars. The label is:
   - `down` if the average future return is below `-0.0002`
   - `neutral` if it lies between `-0.0002` and `0.0002`
   - `up` if it is above `0.0002`

2. **10-minute multiclass extension, 30-minute horizon.**
   The target is the average future log return over the next 30 minutes, using 10-minute bars, with the same three-class threshold rule.

3. **10-minute multiclass extension, 60-minute horizon.**
   The target is the average future log return over the next 60 minutes, using 10-minute bars, again with the same three-class threshold rule.

### Sequence construction

Each training sample uses a fixed lookback window of 300 minutes. This corresponds to:
- 60 bars in the 5-minute baseline
- 30 bars in the 10-minute extensions

Input windows and prediction horizons are required to remain within the same trading day, so cross-day samples are excluded.

### Models and training

We retain the same model families as the earlier notebook: TCN, DLinear-style MLP baseline, and LSTM. To isolate the effect of label construction, class weighting is turned off by default, in line with the advisor feedback. Models are selected by validation macro F1, and final reporting focuses on macro F1, macro AUC, and per-class F1 rather than overall accuracy alone.

### Evaluation

For each asset-model-experiment combination, we report:
- validation and test macro F1
- validation and test macro AUC
- per-class precision, recall, F1, and AUC
- confusion matrix

This evaluation design makes it easier to judge whether gains come from better balance across classes rather than from over-predicting the majority class.